# 03. デルファイ法（Delphi Method） — 練習問題

**対象技術**: AI（生成AI・機械学習・自動化を含む）

匿名・反復・統制されたフィードバックの3原則により専門家集団の判断を収束させる予測手法である。各ラウンドで中央値・四分位範囲（IQR）を集約して全員に返し、回答者が見積もりを修正する過程を繰り返す。収束する問いは合意項目、収束しない問いはシナリオ・プランニングの不確実性軸の候補となる。

`numpy` で回答分布をシミュレートし、`matplotlib` でラウンドごとの収束を可視化する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

RNG = np.random.default_rng(20410)  # 再現性のため固定シード

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## パラメータ定義

15名の専門家パネルが2つの問いに回答する。各問いには参照点となる真値の目安と初期ばらつきを設定する。中央値への引き寄せ係数 `PULL` は、問2を「専門職代替の見立てが割れて収束しにくい」設定にしてある。

In [ ]:
N_EXPERTS = 15
N_ROUNDS = 4

# 問いの定義: (名称, 真値の目安, 初期ばらつきの標準偏差)
QUESTIONS = {
    "専門職タスク人間水準到達年":   {"center": 2032.0, "spread": 6.0},
    "ホワイトカラー業務30%代替年": {"center": 2035.0, "spread": 12.0},
}

# 中央値への引き寄せ係数。問2は代替の見立てが割れ収束しにくい設定にする。
PULL = {
    "専門職タスク人間水準到達年": 0.45,    # よく収束する
    "ホワイトカラー業務30%代替年": 0.12,  # ほとんど収束しない(非合意項目)
}
print(f"パネル {N_EXPERTS} 名 / 最大 {N_ROUNDS} ラウンド")

## 解析関数

四分位の計算、Kendall の一致係数 W の自前実装、1つの問いについて全ラウンドを実行する関数を定義する。

In [ ]:
def quartiles(values):
    """第1四分位・中央値・第3四分位を線形補間で計算する。"""
    v = np.sort(np.asarray(values, dtype=float))
    q1 = np.percentile(v, 25)
    med = np.percentile(v, 50)
    q3 = np.percentile(v, 75)
    return q1, med, q3


def kendall_w(rank_matrix):
    """Kendall の一致係数 W を自前実装する。

    rank_matrix: shape (m, n)。m=評価者数, n=評価対象数。
    W は 0(不一致)〜1(完全一致)。
    """
    m, n = rank_matrix.shape
    rank_sums = rank_matrix.sum(axis=0)
    mean_rs = rank_sums.mean()
    S = np.sum((rank_sums - mean_rs) ** 2)
    denom = m ** 2 * (n ** 3 - n)
    if denom == 0:
        return 1.0
    return 12.0 * S / denom


def run_delphi(name, params, pull):
    """1つの問いについてデルファイの全ラウンドを実行する。"""
    answers = RNG.normal(params["center"], params["spread"], N_EXPERTS)
    history = []
    for rnd in range(1, N_ROUNDS + 1):
        q1, med, q3 = quartiles(answers)
        iqr = q3 - q1
        history.append((rnd, q1, med, q3, iqr))
        if rnd < N_ROUNDS:
            noise = RNG.normal(0.0, params["spread"] * 0.15, N_EXPERTS)
            answers = answers + pull * (med - answers) + noise
    return history, answers

## デルファイ・ラウンドの実行

各問いについて全ラウンドを実行し、ラウンドごとの Q1・中央値・Q3・IQR幅の推移を表示する。

In [ ]:
histories = {}
final_answers = {}
for name, params in QUESTIONS.items():
    history, answers = run_delphi(name, params, PULL[name])
    histories[name] = history
    final_answers[name] = answers

    print(f"■ 問い: 「{name}」")
    print(f"  {'Rnd':>3} | {'Q1':>8} | {'中央値':>8} | {'Q3':>8} | {'IQR幅':>7}")
    print("  " + "-" * 48)
    for rnd, q1, med, q3, iqr in history:
        print(f"  {rnd:>3} | {q1:>8.1f} | {med:>8.1f} | {q3:>8.1f} | {iqr:>7.1f}")
    first_iqr = history[0][4]
    last_iqr = history[-1][4]
    shrink = (1 - last_iqr / first_iqr) * 100 if first_iqr > 0 else 0
    verdict = "合意項目" if last_iqr < 7 else "非合意項目(シナリオ軸の候補)"
    print(f"  IQR幅 {first_iqr:.1f} -> {last_iqr:.1f} ({shrink:.0f}% 縮小)  => {verdict}")
    print()

## 合意度 Kendall の W

各専門家が「2つの問いの年（＝遅さ）」に付ける順位の一致度をKendall の W で測る。

In [ ]:
names = list(QUESTIONS.keys())
mat = np.column_stack([final_answers[n] for n in names])  # (専門家, 問い)
ranks = np.empty_like(mat)
for i in range(mat.shape[0]):
    ranks[i] = np.argsort(np.argsort(mat[i])) + 1
W = kendall_w(ranks)
print(f"対象: {names}")
print(f"W = {W:.3f}  (1に近いほど専門家間で順位が一致)")
print()
print("[解釈] 専門職タスク人間水準到達年は IQR が縮小し合意項目となる。")
print("       一方、ホワイトカラー業務30%代替年は代替の見立てが割れ")
print("       収束せず、非合意項目としてシナリオ・プランニングの")
print("       不確実性軸の根拠になる。")

## 可視化: ラウンドごとの収束

横軸にラウンド、縦軸に予測年をとり、中央値を折れ線で、IQR（Q1〜Q3）を帯（`fill_between`）で描く。合意項目（専門職タスク人間水準到達年）と非合意項目（ホワイトカラー業務30%代替年）を並べ、収束の差を比較する。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
colors = {"専門職タスク人間水準到達年": "#3a6ea5",
          "ホワイトカラー業務30%代替年": "#d1495b"}
titles = {"専門職タスク人間水準到達年":
          "Q1: year AI reaches human-expert level (consensus)",
          "ホワイトカラー業務30%代替年":
          "Q2: year 30%% of white-collar work automated (no consensus)"}

for ax, name in zip(axes, names):
    hist = histories[name]
    rounds = [h[0] for h in hist]
    q1s = np.array([h[1] for h in hist])
    meds = np.array([h[2] for h in hist])
    q3s = np.array([h[3] for h in hist])
    c = colors[name]

    ax.fill_between(rounds, q1s, q3s, color=c, alpha=0.25,
                    label="IQR (Q1-Q3)")
    ax.plot(rounds, meds, "o-", color=c, lw=2.2, markersize=8,
            label="median")
    ax.plot(rounds, q1s, "--", color=c, lw=1, alpha=0.7)
    ax.plot(rounds, q3s, "--", color=c, lw=1, alpha=0.7)

    for r, m, q1, q3 in zip(rounds, meds, q1s, q3s):
        ax.annotate(f"IQR={q3-q1:.1f}", (r, q3), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=8, color=c)

    ax.set_title(titles[name], fontsize=11)
    ax.set_xlabel("Delphi round")
    ax.set_ylabel("predicted year")
    ax.set_xticks(rounds)
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=9)

fig.suptitle("Delphi Method: convergence of expert estimates",
             fontsize=13)
plt.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

デルファイ法は、未来デザイン論文において、専門家パネルへの匿名・反復調査を通じて、ある事象の発生時期や規模についての見積もりを収束させる装置として用いられる。論文は典型的に、各ラウンドで回答の中央値と四分位範囲（IQR）を集約して全員にフィードバックし、回答者が見積もりを修正する過程を繰り返したうえで、最終的な中心推定値と、その周りのばらつき＝合意度を提示する。結論として読者に手渡されるのは、点推定とそれに付された合意の強さという二層の情報である。

この手法がもたらす結論の型は「中心推定値＋合意度」であり、未来は基本的に専門家の集合知によって推定可能な対象として扱われる。時間観の経路で見れば、ここでの未来は過去と専門家の経験の延長線上にあり、収束の過程そのものが「もっともらしい未来は一つに絞り込める」という前提を体現している。境界設定の経路では、結論はパネルに招かれた専門家の知識範囲によって規定され、パネルの構成が見えない領域は推定の対象にすらならない。価値の所在は、誰を専門家として選ぶか、どの問いを立てるかというパネル設計に埋め込まれる。

同時に、この手法は構造的な保守バイアスを結論に持ち込む。反復とフィードバックという仕組みは合意形成を促す一方で、回答者を多数派・主流見解へと引き寄せる収束圧力としても働き、分布のテール——少数の専門家が予見する極端なシナリオや不連続な変化——は外れ値として抑圧されやすい。その結果、結論は穏当で漸進的な方向に寄り、急激なブレークスルーや突発的な崩壊といった非線形の未来は過小評価されがちである。この手法を採る論文は、合意の強さを未来の確からしさと取り違えないこと、そして収束しなかった問いを失敗ではなくシナリオ分岐の所在として積極的に報告することで、保守バイアスを部分的に補正できる。

## 発展課題

**課題A**: IQR の縮小率（前ラウンド比）を見て、一定割合を下回ったら打ち切る収束基準を設計・実装せよ。問いごとに収束ラウンド数が変わるはずである。

**課題B**: 中央値へ寄らない「頑固な外れ値専門家」を `ANCHOR_RESISTANT` として1〜2名混ぜ、収束の遅れと W の低下を定量的に分析せよ。